# CICIDS2017 Network Traffic Preprocessing
**Source:** Canadian Institute for Cybersecurity (CIC)  
**Attack Types:** BENIGN, DoS Hulk, DoS GoldenEye, DoS Slowloris, DoS Slowhttptest, DDoS, PortScan, FTP-Patator, SSH-Patator, Web Attack, Bot, Infiltration, Heartbleed  
**Output:** X_train.npy, X_test.npy, y_train.csv, y_test.csv

## Dataset Domain Knowledge — CICIDS2017 Network Traffic

### How Network Traffic Works
CICIDS2017 captures network traffic between computers in a lab environment.
Features are extracted from network flows — groups of packets between two devices.
This is the most widely used IDS benchmark dataset in research.

### What Each Column Means

| Column | Meaning | Attack Signal |
|--------|---------|---------------|
| `Flow Duration` | How long the connection lasted | Very short = scanning |
| `Total Fwd Packets` | Packets sent forward | Very high = flooding |
| `Fwd Packet Length` | Size of forward packets | Abnormal size = attack |
| `Bwd Packets/s` | Backward packet rate | Very high = DDoS |
| `Destination Port` | Target port number | Known attack ports |
| `Flow Bytes/s` | Data transfer rate | Very high = DoS |
| `SYN Flag Count` | TCP connection requests | High = SYN flood |

### The 12 Attack Types

**DoS Hulk** — floods HTTP requests to overwhelm web server
**DoS GoldenEye** — keeps HTTP connections open until server runs out
**DoS Slowloris** — sends partial requests to exhaust server connections
**DoS Slowhttptest** — slow HTTP attack draining server resources
**DDoS** — coordinated flood from many sources simultaneously
**PortScan** — probes many ports to find vulnerabilities
**FTP-Patator** — brute forces FTP login credentials
**SSH-Patator** — brute forces SSH login credentials
**Web Attack** — SQL injection, XSS, and brute force on web apps
**Bot** — automated malware communicating with command server
**Infiltration** — attacker already inside network moving laterally
**Heartbleed** — exploits OpenSSL vulnerability to steal memory data

### Expected XAI Findings
- **SHAP on Destination Port** → key for detecting PortScan and service attacks
- **SHAP on Bwd Packets/s** → key for DoS/DDoS detection
- **LIME** → local explanation per flow, may highlight different features than SHAP
- **Permutation Importance** → confirms which features cause biggest F1 drop
- **Class imbalance effect** → BENIGN dominates at 80.3%, rare attacks may have near-zero F1
- **Key question** → do SHAP, LIME, and PI agree on top features across 12 attack types?

In [2]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import glob

print("Libraries imported!")

Libraries imported!


In [3]:
# Step 2: Find CICIDS2017 files
import os

# Check common locations
home = os.path.expanduser("~")
print("Looking for CICIDS2017 files...")
print(f"Home folder: {home}")

# List Downloads folder
downloads = os.path.join(home, "Downloads")
files = os.listdir(downloads)
print("\nFiles in Downloads:")
for f in files:
    if ".csv" in f.lower() or "cicids" in f.lower() or "ids" in f.lower():
        print(f"  {f}")

Looking for CICIDS2017 files...
Home folder: /Users/miuyanhong

Files in Downloads:


In [5]:
# Step 3: Search entire Mac for CICIDS files
import subprocess

result = subprocess.run(
    ["find", "/Users/miuyanhong", 
     "-name", "*.csv", 
     "-type", "f"],
    capture_output=True,
    text=True
)

print("CSV files found on the Mac:")
print(result.stdout)

CSV files found on the Mac:
/Users/miuyanhong/tvshows_env/lib/python3.13/site-packages/numpy/_core/tests/data/umath-validation-set-log2.csv
/Users/miuyanhong/tvshows_env/lib/python3.13/site-packages/numpy/_core/tests/data/umath-validation-set-arcsinh.csv
/Users/miuyanhong/tvshows_env/lib/python3.13/site-packages/numpy/_core/tests/data/umath-validation-set-arctanh.csv
/Users/miuyanhong/tvshows_env/lib/python3.13/site-packages/numpy/_core/tests/data/umath-validation-set-sin.csv
/Users/miuyanhong/tvshows_env/lib/python3.13/site-packages/numpy/_core/tests/data/umath-validation-set-cos.csv
/Users/miuyanhong/tvshows_env/lib/python3.13/site-packages/numpy/_core/tests/data/umath-validation-set-cbrt.csv
/Users/miuyanhong/tvshows_env/lib/python3.13/site-packages/numpy/_core/tests/data/umath-validation-set-arctan.csv
/Users/miuyanhong/tvshows_env/lib/python3.13/site-packages/numpy/_core/tests/data/umath-validation-set-cosh.csv
/Users/miuyanhong/tvshows_env/lib/python3.13/site-packages/numpy/_core

In [6]:
# Step 3: Load CICIDS2017 files
import pandas as pd
import glob

# Path to your CICIDS2017 files
path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/cicids_db/*.csv"

# Load all 8 files
files = glob.glob(path)
print(f"Found {len(files)} files:")
for f in files:
    print(f"  {f.split('/')[-1]}")

Found 8 files:
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Monday-WorkingHours.pcap_ISCX.csv
  Friday-WorkingHours-Morning.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Tuesday-WorkingHours.pcap_ISCX.csv
  Wednesday-workingHours.pcap_ISCX.csv
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv


In [6]:
# Step 4: Load and merge all 8 files
import pandas as pd
import glob
import numpy as np

path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/cicids_db/*.csv"
files = glob.glob(path)

print("Loading files... please wait 2-3 minutes")
print("(2.8 million records — takes time!)")

# Load all files
dfs = []
for f in files:
    print(f"Loading: {f.split('/')[-1]}")
    df = pd.read_csv(f, low_memory=False)
    dfs.append(df)

# Merge into one dataframe
df = pd.concat(dfs, ignore_index=True) # concat = merge data 
print(f"\n Total records loaded: {len(df)}")
print(f" Total columns: {len(df.columns)}")

Loading files... please wait 2-3 minutes
(2.8 million records — takes time!)
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv

 Total records loaded: 3119345
 Total columns: 85


In [7]:
# Step 5: Clean the data 
print("Cleaning data...")
print(f"Before cleaning: {len(df)} records") 
# df = dataset(dataframe)

# Remove infiite values
df.replace([np.inf, -np.inf], np.nan, inplace=True) 
# np.inf = infinity, calculation went wrong
# np.nan = not a number = empty or miss cell
# this line: find all infinity values and replace then "ERROR" or "∞" if cells is empty

# Remove NaN values
df.dropna(inplace=True)
# True = edit codument directly; false = edit a photocopy
# na = not available = missing/empty; drop = remove/delete
      
print(f"After cleaning: {len(df)} records")
print(f"Removed: {3119345 - len(df)} records")
print("Cleaning complete!")

Cleaning data...
Before cleaning: 3119345 records
After cleaning: 2827876 records
Removed: 291469 records
Cleaning complete!


In [14]:
# Step 6: Check attack labels 
# what attach types exist in data; how many records per atttach type; is data balances or not

print("Attack types in dataset:")
print(df[' Label'].value_counts())
print(f"\nTotal unique labels: {df[' Label'].nunique()}")

Attack types in dataset:
 Label
BENIGN              2271320
DoS Hulk             230124
PortScan             158804
DDoS                 128025
DoS GoldenEye         10293
FTP-Patator            7935
SSH-Patator            5897
DoS slowloris          5796
DoS Slowhttptest       5499
Web Attack             2180
Bot                    1956
Infiltration             36
Heartbleed               11
Name: count, dtype: int64

Total unique labels: 13


In [12]:
# strip whitespace from all column names
# CICIDS2017 columns have a leading space e.g. ' Label' not 'Label'
df.columns = df.columns.str.strip()
print("Column names cleaned!")
print(f"Label column check: {'Label' in df.columns}")

Column names cleaned!
Label column check: True


In [13]:
# Step 7: Separate features and labels
print("Preparing features and labels...")

# Separate features (X) and labels (y)
# Note: 'Label' has no space after stripping
X = df.drop('Label', axis=1)
y = df['Label'] # label on rightmost (answer/output): BENGIN(normal); DDos(many attacker); Dos(one attacher)
# type of this data for DoS: hulk = floods HTTP requests; GoldEys = keeps connections open; slowlories = sents partial requests; slowwhttptest = slow HTTP attacks

print(f"Features (X): {X.shape}")
print(f"Labels (y):   {y.shape}")
print("Features and labels ready!")

Preparing features and labels...
Features (X): (2827876, 84)
Labels (y):   (2827876,)
Features and labels ready!


In [19]:
# encode text labels to numbers — same as all other datasets
# reason: consistent preprocessing across all 5 datasets
# allows fair comparison in benchmark paper
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Label encoding:")
for i, label in enumerate(le.classes_):
    print(f"  {label} → {i}")

Label encoding:
  BENIGN → 0
  Bot → 1
  DDoS → 2
  DoS GoldenEye → 3
  DoS Hulk → 4
  DoS Slowhttptest → 5
  DoS slowloris → 6
  FTP-Patator → 7
  Heartbleed → 8
  Infiltration → 9
  PortScan → 10
  SSH-Patator → 11
  Web Attack → 12


In [14]:
# Step 8: Remove non-numeric columns and Normalize features(converted the scale number are same unit for fair to comparison)

# Remove non-numeric columns 
print("Checking data types...")

# find non-numeric columns
non_numeric = X.select_dtypes(
    exclude=['float64', 'int64']).columns.tolist()
print(f"Non-numeric columns found: {non_numeric}")

# Remove non-numeric columns
X_clean = X.drop(columns=non_numeric)
print(f"Columns after removing non-numeric: {X_clean.shape[1]}")

# Normalize features
from sklearn.preprocessing import StandardScaler 
# scikit-learn library(ML toolkit); preprocessing = tools for preparing data before training model
# StandardScaler = specific tool that normalized numbers to same scale

print("Normalizing features...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

print(f"Normalization complete!")
print(f"Shape: {X_scaled.shape}")


Checking data types...
Non-numeric columns found: ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
Columns after removing non-numeric: 80
Normalizing features...
Normalization complete!
Shape: (2827876, 80)


In [20]:
# Step 9: Split train / test (70% train, 30 % test) 
# 70 % = model learns form dataset; 30 % model tested on this dataset
# model_selection, stractify and random_state -> professional, fair and reproducible benchmark
from sklearn.model_selection import train_test_split

print("Splitting data...")

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,  y_encoded, 
    test_size=0.30, # testing
    random_state=42, # fixed seed = reproducible 
    stratify = y_encoded,    # keep same % of each attack type (fair split of attacks)  
)

print(f"Training set: {X_train.shape[0]} records")
print(f"Testing set: {X_test.shape[0]} records")
print("Split complete!")

Splitting data...
Training set: 1979513 records
Testing set: 848363 records
Split complete!


In [21]:
# Step 10: Save cleaned data to file
import numpy as np
import os

print("Saving cleaned data...")

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/"
os.makedirs(save_path, exist_ok=True)

# Save all arrays
np.save(save_path + "X_train.npy", X_train)
np.save(save_path + "X_test.npy", X_test)
np.save(save_path + "X_clean.npy", X_clean)

# Save labels — NOW SAVING ENCODED NUMBERS not text
# ← CHANGED: y_train.to_csv → pd.DataFrame(y_train).to_csv
pd.DataFrame(y_train).to_csv(save_path + "y_train.csv", index=False)
pd.DataFrame(y_test).to_csv(save_path + "y_test.csv", index=False)

# save feature names and label classes
pd.DataFrame(X_clean.columns).to_csv(save_path + "feature_names.csv", index=False)
# ← CHANGED: y.unique() → le.classes_ (saves in correct encoded order)
pd.DataFrame(le.classes_).to_csv(save_path + "label_classes.csv", index=False)

print("feature_names.csv saved")
print("label_classes.csv saved")
print(f"Files saved to: {save_path}")
print("Preprocessing 100% complete!")

Saving cleaned data...
feature_names.csv saved
label_classes.csv saved
Files saved to: /Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/
Preprocessing 100% complete!


## Preprocessing Complete

| Item | Value |
|------|-------|
| Source | Canadian Institute for Cybersecurity — CICIDS2017 |
| Original rows | 3,119,345 |
| After cleaning | 2,827,876 |
| Removed rows | 291,469 (NaN + Inf) |
| Total features | 80 |
| Labels | 13 classes — encoded 0-12 (BENIGN=0 ... Web Attack=12) |
| Class imbalance | Yes — BENIGN 80.3% dominates |
| Train set | 1,979,513 rows |
| Test set | 848,363 rows |
| Split ratio | 70% train / 30% test |
| Output files | X_train.npy, X_test.npy, y_train.csv, y_test.csv, feature_names.csv, label_classes.csv |